# 使用图注意力网络分析球员交互和重要性

本notebook使用图注意力网络(GATs)分析球员在比赛中的交互并确定其重要性。

**适配版本**: 2022 FIFA世界杯数据集

**特点**:
- 支持任意2022世界杯比赛数据
- 灵活的比赛选择
- 批量处理多场比赛

## 1. 导入必要的库

In [ ]:
import pandas as pd
import numpy as np
import os
import pickle
import gc
from tqdm import tqdm

import convert_tracking as ct
import plot_functions as pf
import create_graph as cg
import GNNs.model_training as mt
import GNNs.convert_data as cd
import GNNs.GAT as GAT
import scale_graph as sg
import visualisation

print("✓ 所有库导入成功")

## 2. 配置参数

设置要分析的比赛ID。可以是单场比赛或多场比赛。

In [ ]:
# 方式1: 分析单场比赛
GAME_IDS = ['10517']  # 决赛: 阿根廷 vs 法国

# 方式2: 分析多场比赛（取消注释以使用）
# GAME_IDS = ['10517', '3857', '3859']  # 决赛、季军赛等

# 方式3: 分析所有可用比赛（取消注释以使用）
# GAME_IDS = ct.get_available_games()

print(f"将分析 {len(GAME_IDS)} 场比赛")
print(f"比赛ID: {GAME_IDS}")

## 3. 加载并可视化示例数据

首先加载第一场比赛的数据，进行可视化验证。

In [ ]:
# 使用第一场比赛作为示例
example_game_id = GAME_IDS[0]
print(f"加载示例比赛: {example_game_id}")

# 加载元数据
(home_team_id, away_team_id, home_team_name, away_team_name, home_team_start_left,
 home_roster, away_roster,
 roster_game_home_name_dict, roster_game_home_team_name_dict, roster_game_home_pos_dict,
 roster_game_away_name_dict, roster_game_away_team_name_dict, roster_game_away_pos_dict,
 pitch_x_adjustment, pitch_y_adjustment) = ct.get_metadata(example_game_id)

print(f"\n比赛信息:")
print(f"  主队: {home_team_name}")
print(f"  客队: {away_team_name}")

# 加载处理后的数据
data_dir = f'Data/{example_game_id}'
balls_df = pd.read_csv(f'{data_dir}/balls_{example_game_id}.csv')
events_df = pd.read_csv(f'{data_dir}/events_{example_game_id}.csv')
players_df = pd.read_csv(f'{data_dir}/players_{example_game_id}.csv')

print(f"\n数据统计:")
print(f"  球数据: {len(balls_df)} 帧")
print(f"  事件数据: {len(events_df)} 个事件")
print(f"  球员数据: {len(players_df)} 条记录")

### 可视化球员位置

In [ ]:
# 选择一个示例帧进行可视化
example_frame = events_df['frameNum'].iloc[10]
print(f"可视化帧: {example_frame}")

fig, ax = pf.plot_players_on_pitch(
    players_df[players_df['frameNum'] == example_frame],
    balls_df[balls_df['frameNum'] == example_frame],
    annotate=True,
    show_velocities=True
)

### 创建并可视化图结构

In [ ]:
# 创建图
G = cg.create_normalized_graph_directed(
    players_df, balls_df, events_df, example_frame, home_team_name
)

if G is not None:
    print(f"图结构:")
    print(f"  节点数: {G.number_of_nodes()}")
    print(f"  边数: {G.number_of_edges()}")
    
    # 可视化图
    pf.visualize_graph_on_pitch_AT(G, example_frame)
else:
    print("图创建失败")

## 4. 批量创建图数据集

为所有选定的比赛创建图数据集。

In [ ]:
print(f"为 {len(GAME_IDS)} 场比赛创建图数据集...\n")

graphs = []
frame_numbers = []
game_ids_list = []

for game_id in tqdm(GAME_IDS, desc="处理比赛"):
    try:
        # 加载元数据
        metadata = ct.get_metadata(game_id)
        home_team_name = metadata[3]
        
        # 加载数据
        data_dir = f'Data/{game_id}'
        balls_df = pd.read_csv(f'{data_dir}/balls_{game_id}.csv')
        events_df = pd.read_csv(f'{data_dir}/events_{game_id}.csv')
        players_df = pd.read_csv(f'{data_dir}/players_{game_id}.csv')
        
        # 为每个事件帧创建图
        event_frames = events_df['frameNum'].values
        
        for frameNum in tqdm(event_frames, desc=f"比赛 {game_id}", leave=False):
            G = cg.create_normalized_graph_directed(
                players_df, balls_df, events_df, frameNum, home_team_name
            )
            
            if G is not None:
                graphs.append(G)
                frame_numbers.append(frameNum)
                game_ids_list.append(game_id)
        
        # 清理内存
        del players_df, balls_df, events_df
        gc.collect()
        
    except Exception as e:
        print(f"\n警告: 比赛 {game_id} 处理失败: {e}")
        continue

print(f"\n✓ 图创建完成")
print(f"  总图数: {len(graphs)}")
print(f"  来自 {len(set(game_ids_list))} 场比赛")

## 5. 图特征缩放

In [ ]:
print("缩放图特征...")

# 创建并拟合缩放器
graph_scaler = sg.GraphFeatureScaler()
graph_scaler.fit(graphs)
print("✓ 缩放器已拟合")

# 转换所有图
scaled_graphs = [graph_scaler.transform_graph(G) for G in tqdm(graphs, desc="缩放图")]
print(f"✓ {len(scaled_graphs)} 个图已缩放")

## 6. 转换为PyTorch Geometric格式

In [ ]:
print("转换为PyTorch Geometric格式...")
dataset = cd.prepare_dataset_reception(scaled_graphs)

print(f"✓ 数据集创建完成: {len(dataset)} 个图")
print(f"\n数据集统计:")
print(f"  节点特征维度: {dataset[0].x.shape[1]}")
print(f"  边特征维度: {dataset[0].edge_attr.shape[1]}")

## 7. 训练GAT模型

训练两个模型：
1. GAT模型（带注意力机制）
2. 标准GNN模型（用于对比）

In [ ]:
print("训练GAT模型（带注意力机制）...\n")

model_reception_AT, train_loader_AT, test_loader_AT = mt.train_reception_prediction_AT_model(
    scaled_graphs,
    num_epochs=10,
    batch_size=16,
    hidden_channels=32,
    edge_hidden_channels=16,
    lr=0.001
)

print("\n✓ GAT模型训练完成")

In [ ]:
print("训练标准GNN模型（用于对比）...\n")

model_reception, train_loader, test_loader = mt.train_reception_prediction_model(
    scaled_graphs,
    num_epochs=40,
    batch_size=64,
    hidden_channels=32,
    lr=0.001
)

print("\n✓ 标准GNN模型训练完成")

## 8. 模型预测和分析

In [ ]:
# 在一个示例图上进行预测
test_graph_idx = min(40, len(scaled_graphs) - 1)

print(f"在图 {test_graph_idx} 上进行预测...")
results_df, attention_analysis = visualisation.predict_reception_probabilities(
    model_reception_AT,
    scaled_graphs[test_graph_idx],
    head_indexes=range(2)
)

print("\n接球概率预测（前10名）:")
print(results_df.head(10))

# 显示注意力权重
if len(results_df) > 0:
    top_player = results_df['player_name'].iloc[0]
    print(f"\n{top_player} 的注意力权重（前5）:")
    if top_player in attention_analysis:
        for i, attn in enumerate(attention_analysis[top_player][:5]):
            print(f"  {i+1}. {attn['source_player']}: {attn['attention_weight']:.4f}")

## 9. 保存模型和数据

In [ ]:
import torch

# 创建保存目录
os.makedirs('results', exist_ok=True)

# 保存模型
model_path = 'results/model_reception_AT_2022WC.pth'
torch.save(model_reception_AT.state_dict(), model_path)
print(f"✓ GAT模型已保存到: {model_path}")

model_path2 = 'results/model_reception_2022WC.pth'
torch.save(model_reception.state_dict(), model_path2)
print(f"✓ GNN模型已保存到: {model_path2}")

# 保存缩放器
scaler_path = 'results/graph_scaler_2022WC.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(graph_scaler, f)
print(f"✓ 缩放器已保存到: {scaler_path}")

# 保存图数据
graphs_path = 'results/scaled_graphs_2022WC.pkl'
with open(graphs_path, 'wb') as f:
    pickle.dump({
        'scaled_graphs': scaled_graphs,
        'frame_numbers': frame_numbers,
        'game_ids': game_ids_list
    }, f)
print(f"✓ 图数据已保存到: {graphs_path}")

print("\n所有数据已保存！")

## 总结

本notebook完成了以下工作：

1. ✓ 加载2022世界杯比赛数据
2. ✓ 创建图数据集
3. ✓ 特征缩放
4. ✓ 训练GAT和GNN模型
5. ✓ 模型预测和注意力分析
6. ✓ 保存模型和数据

**下一步**：
- 使用 Visualisation.ipynb 进行交互式可视化分析
- 使用 player_eval.py 进行球员评估
- 使用 Experiments.ipynb 进行深入实验分析